# GTS-BERT baseline for ASTE on 14res + 15res + 16res

A **neural but non-generative** baseline for Aspect Sentiment Triplet Extraction, grounded in
Wu et al. (Findings of EMNLP 2020), "Grid Tagging Scheme for Aspect-oriented Fine-grained
Opinion Extraction" (https://aclanthology.org/2020.findings-emnlp.234/) -- used to give the
main T5-family comparison (`notebooks/train-t5-*-for-aste-*.ipynb`) a second reference point
besides `AsteLookupBaseline` (phi-neural, tra cuu tan suat): this one is a real BERT model,
just not a generative one.

**Idea**: for a sentence of n tokens, tag every word-pair (i, j) with i <= j with one of 6 tags:
`N` (no relation), `A` (both words are in the same aspect span), `O` (both words are in the
same opinion span), or `POS`/`NEU`/`NEG` (word i is in an aspect span, word j is in an opinion
span, and together they form a triplet with that sentiment). BERT encodes the sentence, a small
MLP classifies every pair -> an n x n grid; triplets are recovered by reading spans off the
diagonal and majority-voting the sentiment tag between each aspect/opinion span pair.

The grid-building/decoding logic below is copied **verbatim** from `src/data/gts_grid.py`
(tested locally in `tests/test_gts_grid.py`, 10 tests, no GPU needed) -- same
local-logic-tested-then-inlined-into-notebook pattern as the T5 notebooks. Only the BERT model
itself (`src/baseline/gts_bert_model.py`) needs GPU, so it's copied in below too.

**Note on scope**: this is an independent reimplementation grounded in the paper's description
(Section 2.2-2.3), not the authors' original code -- the pairwise representation r_ij (Section
3.1) in particular is a plain concat+MLP here, since the paper text didn't fully specify its
attention-layer formula. Hyperparameters (lr=2e-5, standard for full BERT fine-tuning) are
intentionally *not* the T5 notebooks' lr=3e-4 -- that learning rate is far too high for
BERT-scale full fine-tuning and would diverge.

In [ ]:
import importlib.util, subprocess, sys

required = ["transformers", "torch", "scikit-learn", "pandas"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg.replace('-', '_')) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

In [ ]:
import json
import random
import subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Config

In [ ]:
MODEL_NAME = "bert-base-uncased"
SHORT_NAME = "gts-bert"
MAX_LENGTH = 100          # subword length cap; ASTE sentences are short
BATCH_SIZE = 16
EPOCHS = 20
LEARNING_RATE = 2e-5      # standard full-fine-tune LR for BERT (NOT the T5 notebooks' 3e-4)

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORKING_ROOT / f"{SHORT_NAME}-aste-model"
BEST_MODEL_DIR = WORKING_ROOT / f"{SHORT_NAME}-aste-restaurant-best"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Locate 14res / 15res / 16res Files

Same logic as `notebooks/train-t5-base-for-aste-on-14res-15res-16res.ipynb`: prefer a Kaggle
Input dataset if present, otherwise clone the data repo from GitHub.

In [ ]:
DOMAINS = ["14res", "15res", "16res"]
KAGGLE_DOMAIN_DIRS = {
    "14res": INPUT_ROOT / "semi-triple-14res",
    "15res": INPUT_ROOT / "semi-triple-15res",
    "16res": INPUT_ROOT / "semi-triple-16res",
}

def find_domain_dir(root: Path, domain: str):
    if not root.exists():
        return None
    candidates = []
    explicit_dir = KAGGLE_DOMAIN_DIRS.get(domain)
    if explicit_dir is not None and explicit_dir.exists() and any(explicit_dir.glob("*.txt")):
        candidates.append(explicit_dir)
    for p in root.rglob("*"):
        if p.is_dir() and domain in p.name.lower() and any(p.glob("*.txt")):
            candidates.append(p)
    return sorted(candidates)[0] if candidates else None

def find_split_file(domain_dir: Path, split: str):
    files = sorted(domain_dir.glob("*.txt"))
    names = [f.name.lower() for f in files]
    if split == "train":
        preferred = ["train.txt", f"{domain_dir.name}_train.txt", f"{domain_dir.name}t_train.txt", f"{domain_dir.name}rest_train.txt"]
        patterns = ["train"]
    elif split == "dev":
        preferred = ["dev.txt", "val.txt", f"{domain_dir.name}_dev.txt", f"{domain_dir.name}t_dev.txt", f"{domain_dir.name}rest_dev.txt"]
        patterns = ["dev", "val"]
    elif split == "test":
        preferred = ["test.txt", f"{domain_dir.name}_test.txt", f"{domain_dir.name}t_test.txt", f"{domain_dir.name}rest_test.txt"]
        patterns = ["test"]
    else:
        raise ValueError(split)
    for name in preferred:
        for f in files:
            if f.name.lower() == name.lower():
                return f
    for f, name in zip(files, names):
        if any(pat in name for pat in patterns):
            return f
    return None

def collect_dataset_files(root: Path):
    found = {}
    for domain in DOMAINS:
        d = find_domain_dir(root, domain)
        if d is None:
            continue
        split_files = {split: find_split_file(d, split) for split in ["train", "dev", "test"]}
        if all(split_files.values()):
            found[domain] = split_files
    return found

dataset_files = collect_dataset_files(INPUT_ROOT)
if len(dataset_files) < 3:
    repo_dir = WORKING_ROOT / "SemEval-Triplet-data"
    if not repo_dir.exists():
        subprocess.check_call([
            "git", "clone", "--depth", "1",
            "https://github.com/xuuuluuu/SemEval-Triplet-data.git",
            str(repo_dir),
        ])
    dataset_files = collect_dataset_files(repo_dir)

print(json.dumps({d: {s: str(p) for s, p in splits.items()} for d, splits in dataset_files.items()}, indent=2))
missing_domains = [d for d in DOMAINS if d not in dataset_files]
assert not missing_domains, f"Missing domains: {missing_domains}. Add dataset to Kaggle Input or enable Internet."

## 3. Grid Tagging Scheme -- parsing, grid build/decode

Copied verbatim from `src/data/gts_grid.py` (tested locally: `tests/test_gts_grid.py`, 10
tests, including a round-trip on the exact worked example from the GTS paper's Figure 2/3).

In [ ]:
TAGS = ["N", "A", "O", "POS", "NEU", "NEG"]
TAG2ID = {tag: idx for idx, tag in enumerate(TAGS)}
ID2TAG = {idx: tag for tag, idx in TAG2ID.items()}
IGNORE_INDEX = -100
_SENTIMENT_TO_TAG = {"positive": "POS", "neutral": "NEU", "negative": "NEG"}
_TAG_TO_SENTIMENT = {tag: sentiment for sentiment, tag in _SENTIMENT_TO_TAG.items()}

def _split_token_tag(item):
    token, tag = item.rsplit("=", 1)
    return token, tag

def parse_aste_line_with_spans(line: str):
    parts = line.strip().split("####")
    if len(parts) != 3:
        return None
    sentence, target_tag_text, opinion_tag_text = parts
    tokens = sentence.strip().split()
    target_pairs = [_split_token_tag(item) for item in target_tag_text.strip().split()]
    opinion_pairs = [_split_token_tag(item) for item in opinion_tag_text.strip().split()]

    target_groups = {}
    for idx, (_, tag) in enumerate(target_pairs):
        if tag == "O" or "-" not in tag:
            continue
        group_id, sentiment_code = tag.split("-", 1)
        info = target_groups.setdefault(group_id, {"indices": [], "sentiment": sentiment_code})
        info["indices"].append(idx)

    opinion_groups = {}
    for idx, (_, tag) in enumerate(opinion_pairs):
        if tag == "O":
            continue
        opinion_groups.setdefault(tag, []).append(idx)

    sentiment_map = {"POS": "positive", "NEG": "negative", "NEU": "neutral"}
    triplets = []
    for group_id, target_info in sorted(target_groups.items(), key=lambda x: (len(x[0]), x[0])):
        opinion_group_id = "S" * len(group_id)
        aspect_idx = target_info["indices"]
        opinion_idx = opinion_groups.get(opinion_group_id, [])
        if not aspect_idx or not opinion_idx:
            continue
        sentiment = sentiment_map.get(target_info["sentiment"], target_info["sentiment"].lower())
        triplets.append({
            "aspect_span": (min(aspect_idx), max(aspect_idx) + 1),
            "opinion_span": (min(opinion_idx), max(opinion_idx) + 1),
            "sentiment": sentiment,
        })
    return {"tokens": tokens, "triplets": triplets}

def build_grid_tags(tokens, triplets):
    n = len(tokens)
    grid = np.full((n, n), IGNORE_INDEX, dtype=np.int64)
    for i in range(n):
        for j in range(i, n):
            grid[i, j] = TAG2ID["N"]

    def mark_span(span, tag):
        start, end = span
        for i in range(start, end):
            for j in range(start, end):
                if i <= j:
                    grid[i, j] = TAG2ID[tag]

    for t in triplets:
        mark_span(t["aspect_span"], "A")
        mark_span(t["opinion_span"], "O")
    for t in triplets:
        tag = _SENTIMENT_TO_TAG[t["sentiment"]]
        a_start, a_end = t["aspect_span"]
        o_start, o_end = t["opinion_span"]
        for ai in range(a_start, a_end):
            for oi in range(o_start, o_end):
                i, j = (ai, oi) if ai <= oi else (oi, ai)
                grid[i, j] = TAG2ID[tag]
    return grid

def _diagonal_spans(grid, tag):
    n = grid.shape[0]
    tag_id = TAG2ID[tag]
    spans = []
    i = 0
    while i < n:
        if grid[i, i] == tag_id:
            j = i
            while j + 1 < n and grid[j + 1, j + 1] == tag_id:
                j += 1
            spans.append((i, j + 1))
            i = j + 1
        else:
            i += 1
    return spans

def decode_grid(grid):
    aspect_spans = _diagonal_spans(grid, "A")
    opinion_spans = _diagonal_spans(grid, "O")
    sentiment_tag_ids = {TAG2ID[t] for t in ("POS", "NEU", "NEG")}
    triplets = []
    for a_start, a_end in aspect_spans:
        for o_start, o_end in opinion_spans:
            votes = Counter()
            for ai in range(a_start, a_end):
                for oi in range(o_start, o_end):
                    i, j = (ai, oi) if ai <= oi else (oi, ai)
                    cell = int(grid[i, j])
                    if cell in sentiment_tag_ids:
                        votes[cell] += 1
            if not votes:
                continue
            best_tag_id, _ = max(votes.items(), key=lambda kv: (kv[1], -kv[0]))
            triplets.append({
                "aspect_span": (a_start, a_end), "opinion_span": (o_start, o_end),
                "sentiment": _TAG_TO_SENTIMENT[ID2TAG[best_tag_id]],
            })
    return triplets

def expand_grid_to_subwords(word_grid, word_ids):
    m = len(word_ids)
    grid = np.full((m, m), IGNORE_INDEX, dtype=np.int64)
    for p in range(m):
        wi = word_ids[p]
        if wi is None:
            continue
        for q in range(p, m):
            wj = word_ids[q]
            if wj is None:
                continue
            i, j = (wi, wj) if wi <= wj else (wj, wi)
            grid[p, q] = word_grid[i, j]
    return grid

def collapse_grid_from_subwords(subword_grid, word_ids):
    first_subword = {}
    for p, wi in enumerate(word_ids):
        if wi is not None and wi not in first_subword:
            first_subword[wi] = p
    num_words = len(first_subword)
    grid = np.full((num_words, num_words), TAG2ID["N"], dtype=np.int64)
    for i in range(num_words):
        for j in range(i, num_words):
            grid[i, j] = subword_grid[first_subword[i], first_subword[j]]
    return grid

def triplets_to_phrases(tokens, span_triplets):
    def phrase(span):
        return " ".join(tokens[span[0]:span[1]])
    return [
        (phrase(t["aspect_span"]), phrase(t["opinion_span"]), t["sentiment"])
        for t in span_triplets
    ]

## 4. Build Train / Dev / Test examples

In [ ]:
def load_split(domain, split, path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = parse_aste_line_with_spans(line)
            if item is None:
                continue
            rows.append({"domain": domain, "split": split, **item})
    return rows

train_rows, dev_rows, test_rows = [], [], []
for domain, splits in dataset_files.items():
    train_rows += load_split(domain, "train", splits["train"])
    dev_rows += load_split(domain, "dev", splits["dev"])
    test_rows += load_split(domain, "test", splits["test"])

print(f"train={len(train_rows)} dev={len(dev_rows)} test={len(test_rows)}")

## 5. Tokenize + build subword grids

Grids are built at the **word level** from the gold spans, then expanded to subword positions
via `word_ids()` (a word split into multiple WordPiece subwords has every one of its subwords
inherit the word's tags) -- see `tests/test_gts_grid.py::test_expand_and_collapse_grid_round_trips_through_subword_split`
for the local, GPU-free test of this exact mapping.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GTSDataset(TorchDataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        tokens = row["tokens"]
        encoding = tokenizer(
            tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH,
        )
        word_ids = encoding.word_ids()
        word_grid = build_grid_tags(tokens, row["triplets"])
        subword_grid = expand_grid_to_subwords(word_grid, word_ids)
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "labels": subword_grid,
            "word_ids": word_ids,
            "tokens": tokens,
            "triplets": row["triplets"],
        }

def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len, max_len), IGNORE_INDEX, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
        attention_mask[i, :n] = torch.tensor(b["attention_mask"], dtype=torch.long)
        g = b["labels"]
        labels[i, :g.shape[0], :g.shape[1]] = torch.tensor(g, dtype=torch.long)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "raw": batch}

train_ds = GTSDataset(train_rows)
dev_ds = GTSDataset(dev_rows)
test_ds = GTSDataset(test_rows)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 6. Model

Copied from `src/baseline/gts_bert_model.py` (see that file's docstring for the architecture
note: BERT + concat/MLP pairwise classifier, an independent design choice for r_ij since the
paper doesn't fully specify its attention-layer formula).

In [ ]:
class GTSBertModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 3, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, len(TAGS)),
        )

    def forward(self, input_ids, attention_mask):
        hidden_states = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        hidden_states = self.dropout(hidden_states)
        batch, seq_len, hidden_dim = hidden_states.shape
        h_i = hidden_states.unsqueeze(2).expand(batch, seq_len, seq_len, hidden_dim)
        h_j = hidden_states.unsqueeze(1).expand(batch, seq_len, seq_len, hidden_dim)
        pair_repr = torch.cat([h_i, h_j, h_i * h_j], dim=-1)
        return self.classifier(pair_repr)

model = GTSBertModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## 7. Evaluation (triplet-F1, same metric as the T5 notebooks)

In [ ]:
def normalize_triplet_set(triples):
    return {(a.strip().lower(), o.strip().lower(), s) for a, o, s in triples}

def corpus_triplet_prf(pred_triples_per_ex, gold_triples_per_ex):
    tp = pred_total = gold_total = 0
    for preds, golds in zip(pred_triples_per_ex, gold_triples_per_ex):
        p, g = normalize_triplet_set(preds), normalize_triplet_set(golds)
        tp += len(p & g); pred_total += len(p); gold_total += len(g)
    precision = tp / pred_total if pred_total else 0.0
    recall = tp / gold_total if gold_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_golds = [], []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        pred_tags = logits.argmax(dim=-1).cpu().numpy()
        for i, ex in enumerate(batch["raw"]):
            n = len(ex["input_ids"])
            subword_grid = pred_tags[i, :n, :n]
            word_grid = collapse_grid_from_subwords(subword_grid, ex["word_ids"])
            pred_span_triplets = decode_grid(word_grid)
            all_preds.append(triplets_to_phrases(ex["tokens"], pred_span_triplets))
            all_golds.append(triplets_to_phrases(ex["tokens"], ex["triplets"]))
    return corpus_triplet_prf(all_preds, all_golds)

## 8. Train

In [ ]:
best_f1 = -1.0
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits.reshape(-1, len(TAGS)), labels.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_loss = total_loss / len(train_loader)
    dev_p, dev_r, dev_f1 = evaluate(dev_loader)
    history.append({"epoch": epoch, "train_loss": train_loss, "dev_precision": dev_p, "dev_recall": dev_r, "dev_triplet_f1": dev_f1})
    print(f"epoch {epoch:2d}  train_loss={train_loss:.4f}  dev_triplet_f1={dev_f1:.4f}  (P={dev_p:.4f} R={dev_r:.4f})")
    if dev_f1 > best_f1:
        best_f1 = dev_f1
        torch.save(model.state_dict(), OUTPUT_DIR / "best_state_dict.pt")

print("Best dev triplet-F1:", best_f1)
pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

## 9. Evaluate best checkpoint on Test

In [ ]:
model.load_state_dict(torch.load(OUTPUT_DIR / "best_state_dict.pt"))
test_p, test_r, test_f1 = evaluate(test_loader)
print(f"Test triplet-F1={test_f1:.4f}  P={test_p:.4f}  R={test_r:.4f}")

metrics_df = pd.DataFrame([
    {"split": "dev", "precision": dev_p, "recall": dev_r, "triplet_f1": best_f1},
    {"split": "test", "precision": test_p, "recall": test_r, "triplet_f1": test_f1},
])
display(metrics_df)

## 10. Save Best Model

In [ ]:
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), BEST_MODEL_DIR / "pytorch_model.bin")
tokenizer.save_pretrained(BEST_MODEL_DIR)
metrics_df.to_csv(BEST_MODEL_DIR / "metrics.csv", index=False)
print("Saved best model and metrics to:", BEST_MODEL_DIR)

## 11. Try New Sentences

Sanity check on a couple of hand-written sentences, same spirit as the T5 notebooks' final
cell.

In [ ]:
@torch.no_grad()
def predict_aste(sentence):
    tokens = sentence.split()
    encoding = tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    word_ids = encoding.word_ids()
    model.eval()
    logits = model(input_ids=encoding["input_ids"].to(DEVICE), attention_mask=encoding["attention_mask"].to(DEVICE))
    pred_tags = logits.argmax(dim=-1)[0].cpu().numpy()
    n = len(tokens) and max(i for i in word_ids if i is not None) + 1
    subword_grid = pred_tags[: len(word_ids), : len(word_ids)]
    word_grid = collapse_grid_from_subwords(subword_grid, word_ids)
    return triplets_to_phrases(tokens, decode_grid(word_grid))

examples = [
    "The price is reasonable although the service is poor .",
    "The food was delicious but the wait was long .",
]
for sent in examples:
    print(sent, "->", predict_aste(sent))